# Compustat: ISIN → GVKEY / CUSIP / Company Name

从 Compustat North America 和 Global 的 security + company 表提取 ISIN 映射。

**输出文件**（合并到已有的映射表，追加新记录）:
- `isin/gvkey.pq` — ISIN → GVKEY
- `isin/cusip.pq` — ISIN → CUSIP
- `isin/company_name.pq` — ISIN → Company Name

所有映射的 `update` 时间戳取源文件的下载时间（文件 mtime）。

In [1]:
import pandas as pd
import os
from datetime import datetime

## 1. 读取源数据

In [7]:
# Compustat NA security — has ISIN, gvkey, CUSIP
na_sec = pd.read_parquet('Z:/dataset/Compustat/d_na/security/security', columns=['ISIN', 'gvkey', 'CUSIP'])
na_sec_mtime = datetime.fromtimestamp(os.path.getmtime('Z:/dataset/Compustat/d_na/security/security/0.pq'))
print(f'NA security: {na_sec.shape}, mtime: {na_sec_mtime.date()}')

NA security: (75916, 3), mtime: 2026-03-12


In [8]:
# Compustat Global security — has ISIN, gvkey, CUSIP
gl_sec = pd.read_parquet('Z:/dataset/Compustat/d_global/security/g_security', columns=['ISIN', 'gvkey', 'CUSIP'])
gl_sec_mtime = datetime.fromtimestamp(os.path.getmtime('Z:/dataset/Compustat/d_global/security/g_security/0.pq'))
print(f'Global security: {gl_sec.shape}, mtime: {gl_sec_mtime.date()}')

Global security: (127357, 3), mtime: 2026-03-11


In [9]:
# Compustat NA company — has gvkey, conm, conml
na_co = pd.read_parquet('Z:/dataset/Compustat/d_na/company/company', columns=['gvkey', 'conm', 'conml'])
na_co_mtime = datetime.fromtimestamp(os.path.getmtime('Z:/dataset/Compustat/d_na/company/company/0.pq'))
print(f'NA company: {na_co.shape}, mtime: {na_co_mtime.date()}')

NA company: (57092, 3), mtime: 2026-03-11


In [10]:
# Compustat Global company — has gvkey, conm, conml
gl_co = pd.read_parquet('Z:/dataset/Compustat/d_global/company/g_company', columns=['gvkey', 'conm', 'conml'])
gl_co_mtime = datetime.fromtimestamp(os.path.getmtime('Z:/dataset/Compustat/d_global/company/g_company/0.pq'))
print(f'Global company: {gl_co.shape}, mtime: {gl_co_mtime.date()}')

Global company: (82017, 3), mtime: 2026-03-11


## 2. 提取 GVKEY 映射

从 NA + Global security 表提取 ISIN → GVKEY。

In [11]:
# 合并 NA + Global security
sec = pd.concat([
    na_sec[['ISIN', 'gvkey']].assign(source='Compustat NA', update=na_sec_mtime),
    gl_sec[['ISIN', 'gvkey']].assign(source='Compustat Global', update=gl_sec_mtime),
], ignore_index=True)

# 统一列名为小写，避免与旧数据 concat 后出现重复列
sec.columns = sec.columns.str.lower()

# 去空、去重（同一 isin+gvkey+source 保留第一条）
sec = sec.dropna(subset=['isin', 'gvkey'])
sec['isin'] = sec['isin'].str.strip()
sec = sec[sec['isin'] != '']

# 读取已有 gvkey.pq
gvkey_old = pd.read_parquet('../isin/gvkey.pq')
print(f'Old gvkey.pq: {gvkey_old.shape}')
print(f'Old update range: {gvkey_old["update"].min()} ~ {gvkey_old["update"].max()}')
print(f'Old sources: {gvkey_old["source"].value_counts().to_dict()}')

# 只保留 Compustat NA / Global 来源的旧记录用于去重
other_sources = gvkey_old[~gvkey_old['source'].isin(['Compustat NA', 'Compustat Global'])]

# 新 Compustat 数据去重
sec_dedup = sec.drop_duplicates(subset=['isin', 'gvkey', 'source'])
print(f'New Compustat gvkey records: {len(sec_dedup):,}')

# 合并：其他来源 + 新 Compustat
gvkey_new = pd.concat([other_sources, sec_dedup], ignore_index=True)
gvkey_new = gvkey_new.drop_duplicates(subset=['isin', 'gvkey'], keep='last')
gvkey_new = gvkey_new[['isin', 'gvkey', 'source', 'update']].sort_values(['isin', 'gvkey']).reset_index(drop=True)

print(f'New gvkey.pq: {gvkey_new.shape}')
print(f'New sources: {gvkey_new["source"].value_counts().to_dict()}')
gvkey_new.head()

Old gvkey.pq: (3154733, 4)
Old update range: 2022-06-16 00:00:00 ~ 2022-06-18 00:00:00
Old sources: {'Capital IQ': 3033476, 'Compustat Global': 73549, 'Compustat NA': 47708}
New Compustat gvkey records: 148,295
New gvkey.pq: (3177750, 4)
New sources: {'Capital IQ': 3031572, 'Compustat Global': 93407, 'Compustat NA': 52771}


,isin,gvkey,source,update
0,AE0001651233,274898,Capital IQ,2022-06-18
1,AE0005789484,260795,Capital IQ,2022-06-18
2,AE0005802527,275784,Capital IQ,2022-06-18
3,AE0005802535,260796,Capital IQ,2022-06-18
4,AE0005802543,274891,Capital IQ,2022-06-18


In [13]:
# gvkey_new.to_parquet('../isin/gvkey.pq', index=False)

## 3. 提取 CUSIP 映射

从 NA + Global security 表提取 ISIN → CUSIP。

In [17]:
# 合并 NA + Global security
cusip_sec = pd.concat([
    na_sec[['ISIN', 'CUSIP']].assign(source='Compustat NA', update=na_sec_mtime),
    gl_sec[['ISIN', 'CUSIP']].assign(source='Compustat Global', update=gl_sec_mtime),
], ignore_index=True)

# 统一列名为小写，避免与旧数据 concat 后出现重复列
cusip_sec.columns = cusip_sec.columns.str.lower()

cusip_sec = cusip_sec.dropna(subset=['isin', 'cusip'])
cusip_sec['isin'] = cusip_sec['isin'].str.strip()
cusip_sec['cusip'] = cusip_sec['cusip'].str.strip()
cusip_sec = cusip_sec[(cusip_sec['isin'] != '') & (cusip_sec['cusip'] != '')]

# 读取已有 cusip.pq
cusip_old = pd.read_parquet('../isin/cusip.pq')
print(f'Old cusip.pq: {cusip_old.shape}')
print(f'Old update range: {cusip_old["update"].min()} ~ {cusip_old["update"].max()}')
print(f'Old sources: {cusip_old["source"].value_counts().to_dict()}')

# 只保留非 Compustat 来源的旧记录
other_sources = cusip_old[~cusip_old['source'].isin(['Compustat NA', 'Compustat Global'])]

# 新 Compustat 数据去重
cusip_dedup = cusip_sec.drop_duplicates(subset=['isin', 'cusip', 'source'])
print(f'New Compustat cusip records: {len(cusip_dedup):,}')

# 合并
cusip_new = pd.concat([other_sources, cusip_dedup], ignore_index=True)
cusip_new = cusip_new.drop_duplicates(subset=['isin', 'cusip'], keep='last')
cusip_new = cusip_new[['isin', 'cusip', 'source', 'update']].sort_values(['isin', 'cusip']).reset_index(drop=True)

print(f'New cusip.pq: {cusip_new.shape}')
print(f'New sources: {cusip_new["source"].value_counts().to_dict()}')
cusip_new.head()

Old cusip.pq: (620502, 4)
Old update range: 2022-06-16 00:00:00 ~ 2024-05-05 00:00:00
Old sources: {'FISD': 561091, 'Compustat NA': 34531, 'Audit Analytics': 14661, 'Refinitiv ESG': 10219}
New Compustat cusip records: 54,888
New cusip.pq: (627361, 4)
New sources: {'FISD': 558488, 'Compustat NA': 54888, 'Refinitiv ESG': 10219, 'Audit Analytics': 3766}


,isin,cusip,source,update
0,AE000A40HL14,M4R68K103,Compustat NA,2026-03-12 13:10:49.307499
1,AEDFXA0M6V00,M2851K10,Refinitiv ESG,2022-06-17 00:00:00.000000
2,AGP168751099,P41758106,Compustat NA,2026-03-12 13:10:49.307499
3,AGP8696W1045,P8696W10,Refinitiv ESG,2022-06-17 00:00:00.000000
4,AGP8696W1045,P8696W104,Compustat NA,2026-03-12 13:10:49.307499


In [ ]:
# cusip_new.to_parquet('../isin/cusip.pq', index=False)

## 4. 提取 Company Name 映射

NA / Global security 有 ISIN + gvkey，company 表有 gvkey + conm / conml。
通过 gvkey join 获得从 Compustat 维护的公司名称。

- `conm` = 标准公司名（通常大写）
- `conml` = 公司名全称（含大小写混合）

In [19]:
# ISIN → gvkey (from security tables)
sec_all = pd.concat([
    na_sec[['ISIN', 'gvkey']].assign(source='Compustat NA'),
    gl_sec[['ISIN', 'gvkey']].assign(source='Compustat Global'),
], ignore_index=True)
sec_all = sec_all.dropna(subset=['ISIN', 'gvkey'])
sec_all['ISIN'] = sec_all['ISIN'].str.strip()
sec_all = sec_all[sec_all['ISIN'] != '']

# gvkey → company name (from company tables)
co_all = pd.concat([
    na_co[['gvkey', 'conm', 'conml']],
    gl_co[['gvkey', 'conm', 'conml']],
], ignore_index=True)
co_all = co_all.drop_duplicates(subset=['gvkey'])  # 一个 gvkey 对应一个公司名
print(f'Unique gvkey→name: {len(co_all):,}')

# Join
cn = pd.merge(sec_all, co_all, on='gvkey', how='inner')
# 取 conml 优先（更完整），没有的话用 conm
cn['company_name'] = cn['conml'].where(cn['conml'].notna(), cn['conm'])
cn = cn.dropna(subset=['company_name'])
cn = cn[['ISIN', 'company_name', 'source']].drop_duplicates(subset=['ISIN', 'company_name', 'source'])
print(f'ISIN→company_name from Compustat: {len(cn):,}')
cn.head()

Unique gvkey→name: 136,265
ISIN→company_name from Compustat: 148,295


,ISIN,company_name,source
0,US0001651001,A & M Food Services Inc,Compustat NA
1,US0003541002,A.A. Importing Co Inc,Compustat NA
2,US0003611052,AAR Corp,Compustat NA
3,US0007811047,ABS Industries Inc,Compustat NA
4,US0008723092,ACS Enterprises Inc,Compustat NA


In [20]:
# 给 Compustat NA / Global 分别赋对应 mtime
cn['update'] = cn['source'].map({
    'Compustat NA': na_co_mtime,  # company 表的时间
    'Compustat Global': gl_co_mtime,
})

# 读取已有 company_name.pq
cn_old = pd.read_parquet('../isin/company_name.pq')
print(f'Old company_name.pq: {cn_old.shape}')
print(f'Old update range: {cn_old["update"].min()} ~ {cn_old["update"].max()}')
print(f'Old sources: {cn_old["source"].value_counts().to_dict()}')

# 只保留非 Compustat 来源的旧记录
other_sources = cn_old[~cn_old['source'].isin(['Compustat NA', 'Compustat Global'])]

# 合并
cn_new = pd.concat([other_sources, cn.rename(columns={'ISIN': 'isin'})], ignore_index=True)
cn_new = cn_new.drop_duplicates(subset=['isin', 'company_name'], keep='last')
cn_new = cn_new[['isin', 'company_name', 'source', 'update']].sort_values(['isin', 'company_name']).reset_index(drop=True)

print(f'New company_name.pq: {cn_new.shape}')
print(f'New sources: {cn_new["source"].value_counts().to_dict()}')
cn_new.head()

Old company_name.pq: (10770401, 4)
Old update range: 2022-06-16 00:00:00 ~ 2024-03-16 00:00:00
Old sources: {'Capital IQ': 9470006, 'FISD': 559899, 'RepRisk': 267917, 'Refinitiv Datastream': 200143, 'Compustat Global': 97483, 'S&P ESG': 94078, 'Compustat NA': 63558, 'Sustainalytics': 8975, 'Refinitiv ESG': 8342}
New company_name.pq: (10755349, 4)
New sources: {'Capital IQ': 9469958, 'FISD': 559897, 'RepRisk': 267915, 'Refinitiv Datastream': 200040, 'S&P ESG': 94054, 'Compustat Global': 93407, 'Compustat NA': 52771, 'Sustainalytics': 8975, 'Refinitiv ESG': 8332}


,isin,company_name,source,update
0,AE0001651233,NATIONAL GENERAL IN.PSC,Refinitiv Datastream,2022-06-17
1,AE0001651233,NATIONAL GENERAL INSURANCE CO. (P.J.S.C.),Capital IQ,2022-06-18
2,AE0005789484,EM.INCM.PSC,Refinitiv Datastream,2022-06-17
3,AE0005789484,EMIRATES INSURANCE COMPANY P.J.S.C.,Capital IQ,2022-06-18
4,AE0005802527,SHUAA CAPITAL PSC,Refinitiv Datastream,2022-06-17


In [ ]:
# cn_new.to_parquet('../isin/company_name.pq', index=False)